structured output

pydantic structre

You define a Python blueprint (a class inheriting from BaseModel): You list the exact field names and data types you want back (str, int, float, list[str], etc.).

Field(description=...) guides the model: Just like a tool's docstring tells Gemini what a function does, Field(description=...) tells Gemini what value to put into each field.

Automatic type validation: When you call model_with_structure = model.with_structured_output(Movie), LangChain forces the model to return a validated Python Movie object instead of a free-form paragraph—so you can access fields directly like response.title or response.rating.

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

load_dotenv(override=True)

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

# 1. Define your desired output structure using Pydantic BaseModel
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

# 2. Wrap the model with your Pydantic schema
model_with_structure = model.with_structured_output(Movie)

# 3. Invoke the structured model
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)
print("Title:", response.title)
print("Rating:", response.rating)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8
Title: Inception
Rating: 8.8


we can also get message output along with raw message 

In [3]:
class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{\n  "title": "Inception",\n  "year": 2010,\n  "director": "Christopher Nolan",\n  "rating": 8.8\n}', 'extras': {'signature': 'Eo4BCosBAWkUfRNmW1Jjp77VigCsRTF8JDCvgfoxXaS1DolALemIAAVr7gwyUU1fyPqIkSPt8ueL3Ir52fLd8UhoyHGoZiWApctMve12Q0FP/s0tNNHsFT8kkCYYSh/h+xlrEiIFXN0pc+SCZkY9y0/+/9BZBOXJ9qMQE3vfkzZo/9pVPOTBxPXaaFiNqpr9RA=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'}, id='lc_run--01a208b9-dae5-7077-8553-dc6fce6ca1ad-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 41, 'total_tokens': 49, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8),
 'parsing_error': None}

In [5]:
model_with_raw = model.with_structured_output(Movie, include_raw=True)
response = model_with_raw.invoke("Provide details about the movie Inception")
print("Parsed:", response["parsed"])
print("Raw:", response["raw"])

Parsed: title='Inception' year=2010 director='Christopher Nolan' rating=8.8
Raw: content=[{'type': 'text', 'text': '{\n  "title": "Inception",\n  "year": 2010,\n  "director": "Christopher Nolan",\n  "rating": 8.8\n}', 'extras': {'signature': 'Eo4BCosBAWkUfRMl21srO+NP+n5rMS2SkB/CSb/EvmHEibPtWu4O5kb/SN7SJkKPrYzOVMhun0Jj4CB1WwZQ3tG6kVFVGp4CXnsKhS2vlkrCVouVqNK1UnrI/Cb4aYiNF43dQHUZSwOQRTDAWHIe7ZyVTmMO1k7ZgA4i8q3WTM/ElP0LwkdtNg4Hom2IZr9DAw=='}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'} id='lc_run--01a208be-a37f-76c6-ba73-8403ffcd4c84-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 41, 'total_tokens': 49, 'input_token_details': {'cache_read': 0}}


nested pydanted

In [4]:
class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

structured_model = model.with_structured_output(MovieDetails)
response = structured_model.invoke("Provide details about the movie Inception")
print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')] genres=['Action', 'Sci-Fi', 'Thriller'] budget=160.0


type dic

What TypedDict Is
TypedDict comes directly from Python's standard typing module—you don't need an external library like Pydantic.

It defines the exact keys and expected types for a plain Python dictionary (dict).

Annotated[type, ..., "description"] is how you attach field descriptions to a TypedDict so the AI model knows what value to put into each key (replacing Pydantic's Field(description=...)).

Cell 1: Basic TypedDict Schema with Annotated

In [6]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

model = init_chat_model("google_genai:gemini-3.5-flash-lite")

# Define the schema using TypedDict and Annotated
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)

response = model_with_typeddict.invoke("Provide details about the movie Inception")
print(response)
print("Type:", type(response))
print("Title:", response["title"])

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}
Type: <class 'dict'>
Title: Inception


Cell 2: Nested TypedDict Schema (Actor inside MovieDetails)
Just like with Pydantic, you can nest one TypedDict inside another (for example, a movie containing a list of actors):

In [7]:
from typing import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float | None, ..., "Budget in millions USD"]

structured_model = model.with_structured_output(MovieDetails)
response = structured_model.invoke("Provide details about the movie Inception")
print(response)
print("First actor:", response["cast"][0]["name"])

{'title': 'Inception', 'year': 2010, 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Cobb'}, {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'}, {'name': 'Elliot Page', 'role': 'Ariadne'}, {'name': 'Tom Hardy', 'role': 'Eames'}], 'genres': ['Action', 'Sci-Fi', 'Thriller'], 'budget': 160.0}
First actor: Leonardo DiCaprio


dataclass

What @dataclass Is (and How It Differs)
Built into Python (from dataclasses import dataclass): Just like TypedDict, it requires no external library.

Gives you an object with dot-notation attributes: Unlike TypedDict (which returns a dictionary res["name"]), a @dataclass returns a class instance where you access fields with dot notation (res.name), without Pydantic's heavy runtime validation overhead.

Adding descriptions: Python comments (#) inside a @dataclass are ignored by the AI model. To give Gemini field descriptions, wrap the type in Annotated[str, ..., "description"].

In [1]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}
    ]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [2]:
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated
from dataclasses import dataclass

# 1. Pydantic -> Returns validated object with dot notation (res.name)
class ContactPydantic(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

# 2. TypedDict -> Returns plain dictionary (res["name"]), no validation
class ContactTypedDict(TypedDict):
    """Contact information for a person."""
    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str, ..., "The phone number of the person"]

# 3. Dataclass -> Returns object with dot notation (res.name), no validation
@dataclass
class ContactDataclass:
    """Contact information for a person."""
    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str, ..., "The phone number of the person"]

All three are ways to tell the AI what fields you want in its answer, and they differ only in **what Python format you get back** and **whether Python checks the data types**.

* **1. Pydantic (`BaseModel`) — Strict & Validated Object**
* **What you get back:** A Python object where you access fields with a dot (`response.name`).
* **Main difference:** It strictly checks every value at runtime. If `year` is supposed to be a number (`int`) and the model gives text, Pydantic catches it or converts it immediately. Use this for real apps where wrong data types would break your code.


* **2. `TypedDict` — Simple Dictionary**
* **What you get back:** A normal Python dictionary (`{'name': 'John'}`) where you access fields with square brackets (`response["name"]`).
* **Main difference:** It does **not** check data types at runtime—it just gives you a plain dictionary straight from the JSON. Use this when you want simple JSON/dict data with zero extra setup.


* **3. `@dataclass` — Lightweight Object (No Strict Checking)**
* **What you get back:** A Python object where you access fields with a dot (`response.name`), just like Pydantic.
* **Main difference:** Unlike Pydantic, it does **not** strictly validate data types when creating the object, and it uses Python's built-in `dataclasses` module instead of an external library. Use this when you prefer dot notation (`response.name`) over dictionary brackets (`response["name"]`) without Pydantic's extra rules.



**Quick Rule to Remember:**

* Want a **dictionary** (`res["name"]`)? Use **`TypedDict`**.
* Want an **object with dot notation** (`res.name`) and **no extra library**? Use **`@dataclass`**.
* Want an **object with dot notation** (`res.name`) **plus strict type checking**? Use **`Pydantic`**.